In [10]:
"""
Starter pipeline for predicting probability of trial success
Focus condition: Alzheimer Disease
Data source: ClinicalTrials.gov API v2
"""

import requests
import pandas as pd
from typing import List, Dict, Any

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

# 1. Fetch data from Clinical Trials.gov API v2


In [11]:
BASE_URL = "https://clinicaltrials.gov/api/v2/studies"


def fetch_alzheimer_trials(max_pages: int = 5, page_size: int = 100) -> List[Dict[str, Any]]:
    """
    Fetch Alzheimer trials from ClinicalTrials.gov API v2
    """

    # empty dictionary
    all_studies: List[Dict[str, Any]] = []
    next_page_token = None

    fields = ",".join(
        [
            # top level convenience fields
            "NCTId",
            "BriefTitle",
            "OverallStatus",
            "HasResults",
            # protocol section fields
            "protocolSection.identificationModule.nctId",
            "protocolSection.identificationModule.briefTitle",
            "protocolSection.identificationModule.acronym",
            "protocolSection.descriptionModule.briefSummary",
            "protocolSection.conditionsModule.conditions",
            "protocolSection.designModule.phases",
            "protocolSection.designModule.studyType",
            "protocolSection.designModule.enrollmentInfo.count",
            "protocolSection.statusModule.overallStatus",
            "protocolSection.statusModule.startDateStruct.date",
            "protocolSection.statusModule.primaryCompletionDateStruct.date",
        ]
    )

    for page in range(max_pages):
        params = {
            "format": "json",
            "pageSize": page_size,
            "countTotal": "true",
            "query.cond": "Alzheimer Disease",
            "fields": fields,
        }
        if next_page_token:
            params["pageToken"] = next_page_token

        resp = requests.get(BASE_URL, params=params, timeout=60)
        resp.raise_for_status()
        data = resp.json()

        studies = data.get("studies", [])
        if not studies:
            break

        all_studies.extend(studies)

        next_page_token = data.get("nextPageToken")
        if not next_page_token:
            break

    return all_studies


# 2. Convert raw API JSON into a tidy pandas DataFrame


In [3]:
def flatten_study(study: Dict[str, Any]) -> Dict[str, Any]:
    """
    Flatten a single study dict into a simple row
    """

    protocol = study.get("protocolSection", {}) or {}
    ident = protocol.get("identificationModule", {}) or {}
    desc = protocol.get("descriptionModule", {}) or {}
    conds = protocol.get("conditionsModule", {}) or {}
    design = protocol.get("designModule", {}) or {}
    status = protocol.get("statusModule", {}) or {}

    row = {
        "nct_id": ident.get("nctId") or study.get("nctId"),
        "brief_title": ident.get("briefTitle") or study.get("briefTitle"),
        "acronym": ident.get("acronym"),
        "brief_summary": desc.get("briefSummary"),
        "conditions": " | ".join(conds.get("conditions", [])) if conds.get("conditions") else None,
        "phase": " | ".join(design.get("phases", [])) if design.get("phases") else None,
        "study_type": design.get("studyType"),
        "enrollment": None,
        "overall_status": status.get("overallStatus") or study.get("overallStatus"),
        "start_date": None,
        "primary_completion_date": None,
        "has_results": study.get("hasResults"),
    }

    enrollment_info = design.get("enrollmentInfo") or {}
    if "count" in enrollment_info:
        row["enrollment"] = enrollment_info.get("count")

    start_struct = status.get("startDateStruct") or {}
    if "date" in start_struct:
        row["start_date"] = start_struct.get("date")

    primary_struct = status.get("primaryCompletionDateStruct") or {}
    if "date" in primary_struct:
        row["primary_completion_date"] = primary_struct.get("date")

    return row


In [4]:
def build_trials_dataframe(raw_studies: List[Dict[str, Any]]) -> pd.DataFrame:
    """
    Turn list of study dicts into a dataframe
    """

    flattened = [flatten_study(s) for s in raw_studies]
    df = pd.DataFrame(flattened)

    # basic cleaning
    df = df.dropna(subset=["nct_id"]).drop_duplicates(subset=["nct_id"])

    # optional: filter out very early status values
    # df = df[df["overall_status"].notna()]

    return df


# 3. Create Labels: This is a simple heuristic you can refine


In [5]:
def add_success_label(df: pd.DataFrame) -> pd.DataFrame:
    """
    Label success using a simple rule
    success = completed and has reported results
    """

    df = df.copy()
    df["has_results"] = df["has_results"].fillna(False)

    def label_row(row):
        status = str(row.get("overall_status") or "").upper()
        has_res = bool(row.get("has_results"))
        if status == "COMPLETED" and has_res:
            return 1
        return 0

    df["success_label"] = df.apply(label_row, axis=1)
    return df


# Optional utility to merge in an external labeled set
def merge_external_labels(
    df: pd.DataFrame,
    label_csv_path: str,
    id_col: str = "nct_id",
    label_col: str = "label"
) -> pd.DataFrame:
    """
    Merge user labeled data if you have a separate csv
    csv should have columns like nct_id and label
    """

    labels = pd.read_csv(label_csv_path)
    labels[id_col] = labels[id_col].astype(str)
    df = df.copy()
    df[id_col] = df[id_col].astype(str)
    merged = df.merge(labels[[id_col, label_col]], on=id_col, how="inner")
    merged = merged.rename(columns={label_col: "success_label"})
    return merged

# 4. Build sklearn pipeline: text features + categorical features


In [6]:
def build_pipeline() -> Pipeline:
    """
    Build preprocessing and model pipeline
    """

    # combine title and summary into one text field
    # this column will be created before fitting
    text_col = "text_features"
    cat_cols = ["phase", "study_type", "overall_status"]

    text_transformer = TfidfVectorizer(
        max_features=5000,
        ngram_range=(1, 2)
    )

    cat_transformer = OneHotEncoder(handle_unknown="ignore")

    preprocessor = ColumnTransformer(
        transformers=[
            ("text", text_transformer, text_col),
            ("cat", cat_transformer, cat_cols),
        ]
    )

    model = LogisticRegression(
        max_iter=200,
        solver="liblinear"
    )

    clf = Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("model", model),
        ]
    )

    return clf

# 5. Train, cross validate, and tune hyperparameters

In [7]:
def prepare_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create combined text field and basic cleaning
    """

    df = df.copy()

    df["brief_title"] = df["brief_title"].fillna("")
    df["brief_summary"] = df["brief_summary"].fillna("")
    df["phase"] = df["phase"].fillna("UNKNOWN")
    df["study_type"] = df["study_type"].fillna("UNKNOWN")
    df["overall_status"] = df["overall_status"].fillna("UNKNOWN")

    df["text_features"] = df["brief_title"] + " " + df["brief_summary"]

    return df

In [8]:
def run_cv_and_tuning(df: pd.DataFrame):
    """
    Split data, run cross validation, and grid search
    """

    df = prepare_features(df)

    X = df[["text_features", "phase", "study_type", "overall_status"]]
    y = df["success_label"]

    # basic train test split for final eval
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        stratify=y,
        random_state=42,
    )

    pipeline = build_pipeline()

    # cross validation
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1
    )
    print("CV ROC AUC scores:", cv_scores)
    print("CV ROC AUC mean:", cv_scores.mean())

    # hyperparameter tuning
    param_grid = {
        "model__C": [0.1, 1.0, 10.0],
        "model__penalty": ["l1", "l2"],
    }

    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1,
        verbose=1,
    )

    grid.fit(X_train, y_train)

    print("Best params:", grid.best_params_)
    print("Best CV ROC AUC:", grid.best_score_)

    best_model = grid.best_estimator_

    # final evaluation on held out test set
    y_proba = best_model.predict_proba(X_test)[:, 1]
    y_pred = best_model.predict(X_test)

    test_auc = roc_auc_score(y_test, y_proba)
    print("Test ROC AUC:", test_auc)
    print("Classification report:")
    print(classification_report(y_test, y_pred))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_pred))

    # show a few example predictions
    examples = X_test.copy()
    examples["true_label"] = y_test
    examples["pred_label"] = y_pred
    examples["pred_proba"] = y_proba
    print(examples[["text_features", "true_label", "pred_label", "pred_proba"]].head(5))



# 6. Main script entry point

In [9]:
if __name__ == "__main__":
    # step 1 fetch raw studies
    print("Fetching Alzheimer trials from ClinicalTrials.gov API v2")
    raw = fetch_alzheimer_trials(max_pages=5, page_size=100)
    print(f"Fetched {len(raw)} raw studies")

    # step 2 build dataframe
    df_trials = build_trials_dataframe(raw)
    print(f"Dataframe shape before labeling: {df_trials.shape}")
    print(df_trials.head(3))

    # step 3 add heuristic label
    df_labeled = add_success_label(df_trials)
    print("Label distribution:")
    print(df_labeled["success_label"].value_counts())

    # if you have your own labels, comment out the line above
    # and instead do something like
    # df_labeled = merge_external_labels(df_trials, 'my_labels.csv')

    # step 4 training plus eval
    # filter out rows with only one class if that happens
    if df_labeled["success_label"].nunique() < 2:
        raise ValueError("Need both positive and negative labels to train model")

    run_cv_and_tuning(df_labeled)

Fetching Alzheimer trials from ClinicalTrials.gov API v2
Fetched 500 raw studies
Dataframe shape before labeling: (500, 12)
        nct_id                                        brief_title acronym  \
0  NCT01421056  Evaluation of Safety & Tolerability of Multipl...    None   
1  NCT00898807    Citalopram for Agitation in Alzheimer's Disease   CitAD   
2  NCT03656107                       The Cognition and Flow Study    None   

                                       brief_summary  \
0  The purpose of this study is to evaluate the s...   
1  The purpose of this study is to evaluate the s...   
2  About the research\n\nThere are currently 850,...   

                                          conditions   phase      study_type  \
0                                Alzheimer's Disease  PHASE2  INTERVENTIONAL   
1                    Alzheimer's Disease | Agitation  PHASE3  INTERVENTIONAL   
2  Dementia | Alzheimer Disease | Mild Cognitive ...      NA  INTERVENTIONAL   

   enrollment overall

/Users/sromee/Documents/AIPI-520/aipi520_repo/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/sromee/Documents/AIPI-520/aipi520_repo/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/sromee/Documents/AIPI-520/aipi520_repo/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this b